# Association Measures: PMI, Dice and Log-Likelihood

**Computational Text Analysis Pack · Module 05 — Concordance and Collocation**

*Technique:* Choosing an association statistic and knowing what it favours

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/03-comparing-corpora/05c_association_measures.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

There is no single correct measure of collocation strength. There are several,
each with a known bias, and choosing between them is a methods decision you must
be able to defend.

| Measure | Favours | Use when |
|---|---|---|
| **Raw frequency** | common words | almost never on its own |
| **PMI** | rare words, strongly | finding fixed phrases and names |
| **Dice** | balanced pairs | the two words are similarly common |
| **Log-likelihood** | frequent, reliable pairs | you want results that hold up |

Run all four on the same keyword and the top-ten lists barely overlap. That is
not a scandal — it is why you name the measure.

## The tooling

Standard library again — concordancing and collocation are slicing and counting.
**AntConc** and **CQPweb** are the dedicated tools; **#LancsBox**'s GraphColl
does the network version in module 19.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
def cooccurrence(texts, term, span=5):
    near, total = Counter(), Counter()
    term, hits = term.lower(), 0
    for t in texts:
        toks = tokenise(t)
        total.update(toks)
        for i, w in enumerate(toks):
            if w == term:
                hits += 1
                lo, hi = max(0, i - span), min(len(toks), i + span + 1)
                near.update(toks[lo:i] + toks[i + 1:hi])
    return hits, near, total

In [ ]:
def association(hits, near, total, span=5, min_count=10):
    """Four association measures side by side, on the same pairs."""
    n = sum(total.values())
    window = hits * span * 2
    rows = []
    for w, obs in near.items():
        if obs < min_count:
            continue
        f_w = total[w]
        expected = window * f_w / n
        if expected <= 0:
            continue
        pmi = np.log2(obs / expected)
        dice = 2 * obs / (hits + f_w)
        g2 = 2 * (obs * np.log(obs / expected)
                  + (window - obs) * np.log(max(window - obs, 1)
                                            / max(window - expected, 1e-9)))
        rows.append({"word": w, "observed": obs, "corpus_count": f_w,
                     "pmi": pmi, "dice": dice, "log_likelihood": g2})
    return pd.DataFrame(rows)

In [ ]:
TERM = "government"        # ← change this

eng = df[df["language"] == "English"]
hits, near, total = cooccurrence(eng["text"], TERM, span=5)
assoc = association(hits, near, total, span=5, min_count=10)
print(f"'{TERM}': {hits:,} occurrences, {len(assoc):,} candidate collocates")
print()
for measure in ("observed", "pmi", "dice", "log_likelihood"):
    top = assoc.sort_values(measure, ascending=False).head(8)["word"].tolist()
    print(f"{measure:<16}{', '.join(top)}")

## Four lists, one keyword

Read those four rows carefully.

- **observed** is the frequency list wearing a disguise: *the*, *of*, *and*
- **pmi** goes to the other extreme — rare words that happen to appear beside the
  keyword, often parts of a single proper name
- **dice** is more balanced but still favours pairs of similar frequency
- **log_likelihood** gives the list a corpus linguist would recognise

The default advice: **report log-likelihood, use PMI to find fixed phrases, and
never report raw co-occurrence.**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.8), sharey=False)
for ax, measure in zip(axes, ("pmi", "dice", "log_likelihood")):
    top = assoc.sort_values(measure, ascending=False).head(10).iloc[::-1]
    ax.barh(top["word"], top[measure])
    ax.set_title(measure)
fig.suptitle(f"The same collocates of '{TERM}', ranked three ways")
plt.tight_layout(); plt.show()

In [ ]:
# Why PMI behaves as it does: it rewards rarity by construction.
r = np.corrcoef(np.log(assoc["corpus_count"]), assoc["pmi"])[0, 1]
r2 = np.corrcoef(np.log(assoc["corpus_count"]), assoc["log_likelihood"])[0, 1]
print(f"correlation of log(corpus frequency) with PMI            : {r:+.3f}")
print(f"correlation of log(corpus frequency) with log-likelihood : {r2:+.3f}")
print()
print("PMI is strongly negatively correlated with frequency — it is close to")
print("being a rarity detector. That is useful for finding names and idioms and")
print("misleading if you present it as 'the words most associated with X'.")

### 🔧 Your turn 1

Add a PMI floor: keep only pairs where `corpus_count >= 50`, then re-rank by PMI.

This is the standard fix, usually written PMI with a frequency threshold. Does
the list become usable? What did you have to throw away to get there?

### 🔧 Your turn 2

Pick a keyword central to your research and run all four measures.

Then write the methods sentence: which measure, what span, what minimum
frequency, and why. Three clauses. Without them the result is not reproducible.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** With a floor of 50 the PMI list becomes recognisable —
genuinely associated content words rather than fragments of names. What you threw
away is every low-frequency pair, including real ones: a distinctive phrase that
occurs 20 times is invisible now. There is no threshold that keeps both; state
yours and move on.

**Your turn 2.** A complete sentence looks like: *"Collocates were computed within
a ±5-token window, ranked by log-likelihood, restricted to words occurring at
least 25 times in the window and 50 times in the corpus."* Anyone can reproduce
that. "We examined collocates of X" cannot be reproduced at all, and it is what
most published DH papers say.

</details>

## Where this stops

You have finished with individual words. Everything from here treats a whole
document as a single object to be positioned, grouped and compared.

Next: **Module 06 — clustering documents**.